In [2]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram, array_to_latex
from qiskit.result import marginal_distribution
from qiskit.circuit.library import UGate
from math import pi
import random

We can implement the CHSH game, together with the quantum strategy defined above, in Qiskit as follows.

First, here's the definition of the game itself, which allows an arbitrary strategy to be plugged in as an argument.

In [3]:
def chsh_game(strategy):
    # This function runs the CHSH game, using the strategy (a function
    # from two bits to two bits), returning 1 for a win and 0 for a loss.

    # Choose questions x & y randomly.
    x, y = random.randint(0, 1), random.randint(0, 1)

    # Use the strategy to determine a & b.
    a, b = strategy(x, y)

    # Decide if the strategy wins or loses.
    if (a != b) == (x & y):
        return 1 # Win
    return 0 # Lose

Now we'll create a function that outputs a circuit depending on the questions for Alice and Bob. We'll let the qubits have their default names for simplicity, and we'll use the built-in $Ry(θ)$ gate for Alice and Bob's actions.

____

### The \(R_y\) Gate

The Qiskit `ry(θ)` gate rotates a qubit by an angle \(θ\) about the **\(y\)-axis** of the Bloch sphere.

Its matrix representation is

\[
R_y(θ)=
\begin{bmatrix}
\cos(\theta/2) & -\sin(\theta/2)\\
\sin(\theta/2) & \cos(\theta/2)
\end{bmatrix}.
\]

> **Why is there a \(θ/2\)?**  
> In quantum mechanics, single-qubit rotations are represented by unitary operators generated from the Pauli matrices. This naturally introduces a **half-angle**, so a physical rotation of \(θ\) corresponds to the matrix above.

For example,

\[
R_y(π/4)=
\begin{bmatrix}
\cos(\pi/8) & -\sin(\pi/8)\\
\sin(\pi/8) & \cos(\pi/8)
\end{bmatrix},
\]

and

\[
R_y(-π/4)=
\begin{bmatrix}
\cos(\pi/8) & \sin(\pi/8)\\
-\sin(\pi/8) & \cos(\pi/8)
\end{bmatrix}.
\]

Thus, `ry(θ)` changes the measurement basis by rotating the qubit before it is measured in the computational (\(Z\)) basis.